# Room Window: video → 3D walkthrough (Colab)

Runs [arissazheng/projection](https://github.com/arissazheng/projection) on a Colab GPU.
It uses the repo's own frame extraction and COLMAP alignment. The training step uses **gsplat**
(CUDA) instead of Brush, which needs Vulkan graphics drivers that Colab doesn't reliably have.
The result is the same `scene.ply` and `settings.json`, opened in the repo's own walkthrough viewer.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.
Then run the cells top to bottom. Every cell ends with a ✅ or ❌ line.

| Step | Time on a T4 |
|---|---|
| 1–3 Setup + compile gsplat | ~8–12 min (first run only) |
| 4 Add video | ~1 min |
| 5 Align + train | ~10 min alignment, ~10–15 min training |
| 6 Save to Drive | ~1 min |
| 7 Find objects (SAM 3) | ~5–15 min |
| 8 Open viewer | ~1 min |

In [ ]:
#@title 1 · Check GPU
import subprocess, torch
if not torch.cuda.is_available():
    print('❌ STEP 1 FAILED: no GPU. Runtime → Change runtime type → T4 GPU, then rerun.')
else:
    name = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f'✅ STEP 1 DONE: {name}, {mem:.0f} GB, torch {torch.__version__}, CUDA {torch.version.cuda}')

In [ ]:
#@title 2 · Install the repo and dependencies (~2 min)
#@markdown Rerun this cell to pull the newest code. Set BRANCH if the faster version lives on another branch.
REPO_URL = "https://github.com/arissazheng/projection" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
%cd /content
!rm -rf /content/projection && git clone -q --depth 1 -b "$BRANCH" "$REPO_URL" /content/projection && git -C /content/projection log -1 --format='Repo commit: %h %s (%cr)'
!pip install -q "pycolmap==4.2.1" "imageio-ffmpeg==0.8.1" "pillow-heif>=1,<2" "Flask>=3.1,<4" ninja gsplat==1.5.3 "transformers>=5.24,<6" accelerate 2>&1 | grep -v "already satisfied" | tail -3
# The repo's walkthrough viewer (same pinned SuperSplat viewer as scripts/setup_walkthrough.sh)
!mkdir -p /content/projection/.walkthrough-tools/supersplat && cd /content/projection/.walkthrough-tools/supersplat && curl -fsSL https://registry.npmjs.org/@playcanvas/supersplat-viewer/-/supersplat-viewer-1.43.3.tgz | tar -xz
import os, importlib
ok = os.path.isfile('/content/projection/pipeline/reconstruct_room.py') and os.path.isfile('/content/projection/.walkthrough-tools/supersplat/package/public/index.html')
# Extra requirements the updated repo may add (Brush-specific setup is skipped on Colab).
!pip install -q -r /content/projection/requirements-walkthrough.txt 2>&1 | grep -v "already satisfied" | tail -3
# The runner below calls these repo functions; warn clearly if an update renamed any of them.
import sys, inspect
sys.path.insert(0, '/content/projection')
for mod in [m for m in list(sys.modules) if m == 'pipeline' or m.startswith('pipeline.')]:
    del sys.modules[mod]
try:
    import pycolmap, gsplat
    from pipeline import reconstruct_room as rr, walkthrough as wt
    needed = {rr: ['input_kind', 'prepare_images', 'align_images', 'prepare_training_data', 'orient_model'],
              wt: ['create_scene', 'publish_scene', 'create_app']}
    missing = [f'{m.__name__}.{f}' for m, fs in needed.items() for f in fs if not hasattr(m, f)]
    if not ok:
        print('❌ STEP 2 FAILED: repo or viewer files missing')
    elif missing:
        print(f'❌ STEP 2 FAILED: the repo update changed functions this notebook uses: {missing}. The runner needs updating.')
    else:
        print(f'✅ STEP 2 DONE: pycolmap {pycolmap.__version__}, gsplat {gsplat.__version__}, repo API compatible')
except Exception as e:
    print(f'❌ STEP 2 FAILED: {e!r}')

In [ ]:
#@title 3 · Compile gsplat's CUDA kernels (~5–10 min, first run only)
import os, subprocess, torch
major, minor = torch.cuda.get_device_capability(0)
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{major}.{minor}'
os.environ.setdefault('MAX_JOBS', str(os.cpu_count() or 2))
r = subprocess.run(['python', '-c', 'from gsplat.cuda._backend import _C; print(_C is not None)'],
                   capture_output=True, text=True, env=os.environ)
open('/content/gsplat_build.log', 'w').write(r.stdout + r.stderr)
if r.returncode == 0 and 'True' in r.stdout:
    print('✅ STEP 3 DONE: gsplat CUDA kernels ready')
else:
    print((r.stdout + r.stderr)[-2500:])
    print('❌ STEP 3 FAILED: gsplat did not compile (full log: /content/gsplat_build.log)')

In [ ]:
%%writefile /content/colab_room.py
"""Colab runner for Room Window: the repo's own frame extraction + COLMAP alignment,
with Brush's training step replaced by gsplat (CUDA). Produces the same scene folder
(scene.ply + settings.json) the repo's walkthrough server serves.

Usage: python colab_room.py VIDEO --repo /content/projection --out /content/scene [--steps 7000]
"""

from __future__ import annotations

import argparse
import json
import math
import shutil
import sys
import time
from pathlib import Path

import numpy as np

LOG_FILE = Path('/content/progress.log') if Path('/content').exists() else Path('progress.log')


def log(msg: str) -> None:
    line = f'[{time.strftime("%H:%M:%S")}] {msg}'
    print(line, flush=True)
    with LOG_FILE.open('a') as f:
        f.write(line + '\n')


# ---------------------------------------------------------------- data loading

def load_dataset(training: Path):
    """Read the undistorted, oriented COLMAP model written by the repo pipeline."""
    import pycolmap
    from PIL import Image

    model = pycolmap.Reconstruction(training / 'sparse')
    images, viewmats, Ks = [], [], []
    for image_id in sorted(model.reg_image_ids(), key=lambda i: model.image(i).name):
        image = model.image(image_id)
        camera = model.camera(image.camera_id)
        path = training / 'images' / image.name
        if not path.is_file():
            continue
        with Image.open(path) as source:
            rgb = np.asarray(source.convert('RGB'), dtype=np.uint8)
        # Undistorted images may be resized relative to the camera; rescale K to match.
        K = np.array(camera.calibration_matrix(), dtype=np.float64)
        sx, sy = rgb.shape[1] / camera.width, rgb.shape[0] / camera.height
        K[0] *= sx
        K[1] *= sy
        w2c = np.eye(4)
        w2c[:3, :4] = np.array(image.cam_from_world().matrix())
        images.append(rgb)
        viewmats.append(w2c)
        Ks.append(K)

    points = np.array([p.xyz for p in model.points3D.values()], dtype=np.float32)
    colors = np.array([p.color for p in model.points3D.values()], dtype=np.float32) / 255.0
    if len(images) < 3 or len(points) < 10:
        raise ValueError(f'Too little data to train: {len(images)} images, {len(points)} points.')
    return images, np.array(viewmats, np.float32), np.array(Ks, np.float32), points, colors


def knn_scale(points: np.ndarray, k: int = 3) -> np.ndarray:
    from scipy.spatial import cKDTree

    dist, _ = cKDTree(points).query(points, k=k + 1)
    mean = np.sqrt((dist[:, 1:] ** 2).mean(axis=1))
    return np.clip(mean, 1e-4, None)


# ---------------------------------------------------------------- PLY export

SH_C0 = 0.28209479177387814


def write_ply(path: Path, means, scales, quats, opacities, sh0, shN) -> None:
    """Standard 3DGS PLY (what Brush/INRIA/SuperSplat use)."""
    n = means.shape[0]
    f_dc = sh0.reshape(n, 3)
    f_rest = shN.transpose(0, 2, 1).reshape(n, -1)  # channel-major, like INRIA
    quats = quats / np.linalg.norm(quats, axis=1, keepdims=True)
    fields = ['x', 'y', 'z', 'nx', 'ny', 'nz']
    fields += [f'f_dc_{i}' for i in range(3)] + [f'f_rest_{i}' for i in range(f_rest.shape[1])]
    fields += ['opacity'] + [f'scale_{i}' for i in range(3)] + [f'rot_{i}' for i in range(4)]
    data = np.concatenate([means, np.zeros_like(means), f_dc, f_rest, opacities.reshape(n, 1), scales, quats],
                          axis=1).astype('<f4')
    header = 'ply\nformat binary_little_endian 1.0\n' + f'element vertex {n}\n'
    header += ''.join(f'property float {name}\n' for name in fields) + 'end_header\n'
    with open(path, 'wb') as f:
        f.write(header.encode('ascii'))
        f.write(np.ascontiguousarray(data).tobytes())


# ---------------------------------------------------------------- training

def ssim(a, b):
    import torch
    import torch.nn.functional as F

    window = 11
    coords = torch.arange(window, dtype=a.dtype, device=a.device) - window // 2
    g = torch.exp(-coords ** 2 / (2 * 1.5 ** 2))
    g = (g / g.sum())
    kernel = (g[:, None] @ g[None, :]).expand(a.shape[1], 1, window, window).contiguous()
    conv = lambda x: F.conv2d(x, kernel, padding=window // 2, groups=a.shape[1])
    mu_a, mu_b = conv(a), conv(b)
    s_a = conv(a * a) - mu_a ** 2
    s_b = conv(b * b) - mu_b ** 2
    s_ab = conv(a * b) - mu_a * mu_b
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    value = ((2 * mu_a * mu_b + c1) * (2 * s_ab + c2)) / ((mu_a ** 2 + mu_b ** 2 + c1) * (s_a + s_b + c2))
    return value.mean()


def train(training: Path, output: Path, steps: int, sh_degree: int = 3, max_splats: int = 1_500_000,
          device: str = 'cuda') -> int:
    import torch
    from gsplat import rasterization
    from gsplat.strategy import DefaultStrategy

    device = torch.device(device)
    images, viewmats, Ks, points, colors = load_dataset(training)
    log(f'Training on {len(images)} images, {len(points)} starting points, {steps} steps')

    cam_centers = np.linalg.inv(viewmats)[:, :3, 3]
    scene_scale = float(np.linalg.norm(cam_centers - cam_centers.mean(0), axis=1).max()) * 1.1
    scene_scale = max(scene_scale, 1e-3)

    gt = [torch.from_numpy(img).to(device).float().div_(255.0) for img in images]
    viewmats_t = torch.from_numpy(viewmats).to(device)
    Ks_t = torch.from_numpy(Ks).to(device)

    n = len(points)
    num_sh = (sh_degree + 1) ** 2
    rgb = torch.from_numpy(colors)
    params = {
        'means': torch.from_numpy(points),
        'scales': torch.log(torch.from_numpy(knn_scale(points)).float()).unsqueeze(1).repeat(1, 3),
        'quats': torch.nn.functional.normalize(torch.rand(n, 4), dim=-1),
        'opacities': torch.logit(torch.full((n,), 0.1)),
        'sh0': ((rgb - 0.5) / SH_C0).unsqueeze(1),
        'shN': torch.zeros(n, num_sh - 1, 3),
    }
    splats = torch.nn.ParameterDict({k: torch.nn.Parameter(v.float().to(device)) for k, v in params.items()})
    lrs = {'means': 1.6e-4 * scene_scale, 'scales': 5e-3, 'quats': 1e-3, 'opacities': 5e-2,
           'sh0': 2.5e-3, 'shN': 2.5e-3 / 20}
    optimizers = {k: torch.optim.Adam([{'params': splats[k], 'lr': lr, 'name': k}], eps=1e-15)
                  for k, lr in lrs.items()}
    means_sched = torch.optim.lr_scheduler.ExponentialLR(optimizers['means'], gamma=0.01 ** (1.0 / steps))

    strategy = DefaultStrategy(refine_start_iter=500, refine_stop_iter=int(steps * 0.8), refine_every=100,
                               reset_every=3000, verbose=False)
    strategy.check_sanity(splats, optimizers)
    state = strategy.initialize_state(scene_scale=scene_scale)

    rng = np.random.default_rng(0)
    order = []
    start = time.time()
    for step in range(steps):
        if not order:
            order = list(rng.permutation(len(gt)))
        i = order.pop()
        target = gt[i]
        h, w = target.shape[:2]
        renders, _, info = rasterization(
            means=splats['means'], quats=splats['quats'], scales=torch.exp(splats['scales']),
            opacities=torch.sigmoid(splats['opacities']),
            colors=torch.cat([splats['sh0'], splats['shN']], 1),
            viewmats=viewmats_t[i:i + 1], Ks=Ks_t[i:i + 1], width=w, height=h,
            sh_degree=min(step // 1000, sh_degree), packed=False,
        )
        render = renders[0]
        strategy.step_pre_backward(splats, optimizers, state, step, info)
        l1 = (render - target).abs().mean()
        loss = 0.8 * l1 + 0.2 * (1 - ssim(render.permute(2, 0, 1)[None], target.permute(2, 0, 1)[None]))
        loss.backward()
        for opt in optimizers.values():
            opt.step()
            opt.zero_grad(set_to_none=True)
        means_sched.step()
        # Stop growing once the splat budget is used up (keeps T4 memory safe).
        if len(splats['means']) > max_splats and strategy.refine_stop_iter > step:
            strategy.refine_stop_iter = step
            log(f'Reached {max_splats} splats; stopping densification at step {step}')
        strategy.step_post_backward(splats, optimizers, state, step, info, packed=False)

        if step % 500 == 0 or step == steps - 1:
            psnr = -10 * math.log10(max(((render - target) ** 2).mean().item(), 1e-10))
            elapsed = time.time() - start
            eta = elapsed / (step + 1) * (steps - step - 1)
            log(f'step {step + 1}/{steps}  loss {loss.item():.4f}  psnr {psnr:.1f}  '
                f'splats {len(splats["means"]):,}  eta {eta / 60:.1f} min')

    output.mkdir(parents=True, exist_ok=True)
    with torch.no_grad():
        write_ply(output / 'scene.ply', *(splats[k].detach().cpu().numpy() for k in
                                         ('means', 'scales', 'quats', 'opacities', 'sh0', 'shN')))
    return len(splats['means'])


# ---------------------------------------------------------------- full pipeline

def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument('inputs', nargs='+', help='One video, or 12-500 photos')
    parser.add_argument('--repo', default='/content/projection')
    parser.add_argument('--out', default='/content/scene')
    parser.add_argument('--work', default='/content/work')
    parser.add_argument('--steps', type=int, default=7000)
    parser.add_argument('--align-only', action='store_true')
    args = parser.parse_args()

    sys.path.insert(0, args.repo)
    from pipeline import reconstruct_room as rr

    paths = [Path(p) for p in args.inputs]
    work, out = Path(args.work), Path(args.out)
    shutil.rmtree(work, ignore_errors=True)
    shutil.rmtree(out, ignore_errors=True)
    work.mkdir(parents=True)
    dataset = work / 'dataset'
    dataset.mkdir()

    t0 = time.time()
    kind = rr.input_kind(paths)
    report = lambda state, message: log(f'{state}: {message}')
    report('preparing', 'Extracting video frames…' if kind == 'video' else 'Preparing photos…')
    count = rr.prepare_images(paths, kind, dataset / 'images')
    log(f'{count} frames ready')
    model = rr.align_images(dataset, dataset / 'images', kind, count, report)
    log(f'COLMAP aligned {model.num_reg_images()} of {count} images, {model.num_points3D()} points')
    report('preparing', 'Correcting lens distortion in the aligned images…')
    training, model = rr.prepare_training_data(model, dataset / 'images', dataset)
    registered = model.num_reg_images()
    camera = rr.orient_model(model)
    model.write(training / 'sparse')
    log(f'Alignment finished in {(time.time() - t0) / 60:.1f} min')
    (work / 'camera.json').write_text(json.dumps({'camera': camera, 'registered': registered, 'count': count, 'training': str(training)}))
    if args.align_only:
        return

    t1 = time.time()
    n = train(training, out, args.steps)
    log(f'Training finished in {(time.time() - t1) / 60:.1f} min, {n:,} splats')

    # Same settings.json the repo's walkthrough server writes for SuperSplat.
    from pipeline import walkthrough as wt
    scene, base = wt.create_scene(paths[0].stem if kind == 'video' else 'Room photos')
    shutil.copytree(out, scene / 'result_tmp')
    wt.publish_scene(scene, base, scene / 'result_tmp', camera, registered, count)
    (out / 'settings.json').write_text((scene / 'result' / 'settings.json').read_text())
    (out / 'scene_id.txt').write_text(scene.name)
    log(f'DONE scene_id={scene.name}')


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/segment_objects.py
"""Find interactable objects in a trained room scene.

SAM 3 segments + tracks text-prompted objects across the aligned video frames, then
depth-checked voting assigns each Gaussian splat to an object (or background).

Usage: python segment_objects.py --work /content/work --scene /content/scene/scene.ply --out /content/objects
Outputs: objects.json, labels.npy, object_XX_<label>.ply, scene_without_objects.ply, preview.jpg
"""

from __future__ import annotations

import argparse
import json
import pickle
import re
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).parent))
from colab_room import load_dataset, log  # noqa: E402

DEFAULT_PROMPTS = ('mug, cup, bottle, bowl, plate, book, laptop, keyboard, phone, remote control, lamp, '
                   'pillow, blanket, chair, stool, backpack, bag, box, basket, plant, vase, picture frame, '
                   'clock, speaker, shoe, toy')

PLY_TYPES = {'float': 'f4', 'float32': 'f4', 'double': 'f8', 'uchar': 'u1', 'uint8': 'u1', 'char': 'i1',
             'short': 'i2', 'ushort': 'u2', 'int': 'i4', 'int32': 'i4', 'uint': 'u4', 'uint32': 'u4'}


# ---------------------------------------------------------------- PLY I/O (keeps every property)

def read_ply(path: Path):
    data = path.read_bytes()
    end = data.index(b'end_header\n') + len(b'end_header\n')
    lines = data[:end].decode('ascii').splitlines()
    if 'format binary_little_endian 1.0' not in lines:
        raise ValueError(f'{path.name}: only binary little-endian PLY is supported')
    count, fields, in_vertex = 0, [], False
    for line in lines:
        parts = line.split()
        if parts[:2] == ['element', 'vertex']:
            count, in_vertex = int(parts[2]), True
        elif parts[:1] == ['element']:
            in_vertex = False
        elif parts[:1] == ['property'] and in_vertex:
            fields.append((parts[2], '<' + PLY_TYPES[parts[1]]))
    rows = np.frombuffer(data, dtype=np.dtype(fields), count=count, offset=end)
    return rows


def write_ply(path: Path, rows: np.ndarray) -> None:
    inverse = {v: k for k, v in PLY_TYPES.items() if k in ('float', 'double', 'uchar', 'char', 'short', 'ushort', 'int', 'uint')}
    header = ['ply', 'format binary_little_endian 1.0', f'element vertex {len(rows)}']
    header += [f'property {inverse[rows.dtype[name].str[1:]]} {name}' for name in rows.dtype.names]
    with open(path, 'wb') as f:
        f.write(('\n'.join(header) + '\nend_header\n').encode('ascii'))
        f.write(np.ascontiguousarray(rows).tobytes())


# ---------------------------------------------------------------- SAM 3

def run_sam3(frames: list[np.ndarray], prompts: list[str], dtype_name: str, window: int = 40, overlap: int = 8):
    """Segment in overlapping windows (flat GPU memory). Returns per-frame int32 label maps (-1 = none),
    per-track info, and 2D-overlap merge pairs (same object seen twice, within or across windows)."""
    import torch
    from transformers import Sam3VideoModel, Sam3VideoProcessor

    dtype = {'float16': torch.float16, 'float32': torch.float32}[dtype_name]
    model = Sam3VideoModel.from_pretrained('facebook/sam3').to('cuda', dtype=dtype).eval()
    processor = Sam3VideoProcessor.from_pretrained('facebook/sam3')

    n = len(frames)
    h, w = frames[0].shape[:2]
    label_maps = [None] * n
    tracks: dict[int, dict] = {}
    pair_iou: dict[tuple[int, int], float] = {}
    stride = max(window - overlap, 1)
    starts = list(range(0, max(n - overlap, 1), stride))

    for wi, start in enumerate(starts):
        end = min(start + window, n)
        base = wi * 100000  # keeps track ids unique across windows
        session = processor.init_video_session(video=np.stack(frames[start:end]), inference_device='cuda',
                                               processing_device='cpu', video_storage_device='cpu', dtype=dtype)
        session = processor.add_text_prompt(inference_session=session, text=prompts)
        local: dict[int, np.ndarray] = {}
        with torch.inference_mode():
            for out in model.propagate_in_video_iterator(inference_session=session):
                res = processor.postprocess_outputs(session, out)
                i = start + out.frame_idx
                ids = [base + int(x) for x in res['object_ids'].tolist()]
                scores = res['scores'].float()
                if not torch.isfinite(scores).all():
                    raise FloatingPointError('non-finite SAM 3 scores')
                masks = res['masks']
                if masks.ndim == 4:
                    masks = masks[:, 0]
                masks = masks.bool()
                if masks.shape[-2:] != (h, w) and len(ids):
                    masks = torch.nn.functional.interpolate(masks[:, None].float(), size=(h, w), mode='nearest')[:, 0].bool()
                prompt_of = {base + int(oid): text for text, oids in res.get('prompt_to_obj_ids', {}).items() for oid in oids}

                labels = torch.full((h, w), -1, dtype=torch.int32, device=masks.device)
                areas = masks.flatten(1).sum(1) if len(ids) else torch.zeros(0)
                for k in torch.argsort(areas, descending=True).tolist():  # smaller (nested) objects win overlaps
                    if areas[k] == 0:
                        continue
                    labels[masks[k]] = ids[k]
                    t = tracks.setdefault(ids[k], {'prompt': prompt_of.get(ids[k], '?'), 'score': 0.0, 'frames': 0,
                                                   'best_frame': i, 'best_area': 0, 'max_area': 0})
                    t['score'] = max(t['score'], float(scores[k]))
                    t['frames'] += 1
                    t['max_area'] = max(t['max_area'], int(areas[k]))
                    if areas[k] > t['best_area']:
                        t['best_frame'], t['best_area'] = i, int(areas[k])
                # Same physical object found by two prompts (e.g. cup + mug) -> high mask overlap
                if len(ids) > 1:
                    flat = masks.flatten(1).float()
                    inter = flat @ flat.T
                    union = areas[:, None] + areas[None, :] - inter
                    iou = (inter / union.clamp(min=1)).cpu().numpy()
                    for a in range(len(ids)):
                        for b in range(a + 1, len(ids)):
                            if iou[a, b] > 0.5:
                                key = (min(ids[a], ids[b]), max(ids[a], ids[b]))
                                pair_iou[key] = max(pair_iou.get(key, 0.0), float(iou[a, b]))
                local[i] = labels.cpu().numpy()
                if i % 20 == 0:
                    log(f'SAM 3 frame {i + 1}/{n} (window {wi + 1}/{len(starts)}): {len(ids)} objects visible, {len(tracks)} tracks so far')
        del session
        torch.cuda.empty_cache()

        for i, lab in local.items():
            prev = label_maps[i]
            if prev is None:
                label_maps[i] = lab
                continue
            # Overlap frame: link tracks across windows by mask IoU; keep the earlier window's map.
            link_windows(prev, lab, pair_iou)

    del model
    torch.cuda.empty_cache()
    for i, m in enumerate(label_maps):
        if m is None:
            label_maps[i] = np.full((h, w), -1, np.int32)
    return label_maps, tracks, pair_iou


def link_windows(prev: np.ndarray, new: np.ndarray, pair_iou: dict, thresh: float = 0.5) -> None:
    """Record (prev_id, new_id) pairs whose masks overlap with IoU > thresh in the same frame."""
    pids = [p for p in np.unique(prev) if p >= 0]
    nids = [q for q in np.unique(new) if q >= 0]
    for p in pids:
        pm = prev == p
        pa = pm.sum()
        for q in nids:
            qm = new == q
            inter = np.count_nonzero(pm & qm)
            if inter == 0:
                continue
            iou = inter / (pa + qm.sum() - inter)
            if iou > thresh:
                key = (int(min(p, q)), int(max(p, q)))
                pair_iou[key] = max(pair_iou.get(key, 0.0), float(iou))


class UnionFind:
    def __init__(self):
        self.parent = {}

    def find(self, x):
        self.parent.setdefault(x, x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        self.parent[self.find(b)] = self.find(a)


# ---------------------------------------------------------------- 3D voting

def splat_params(rows):
    def stack(prefix, n):
        return np.stack([rows[f'{prefix}{i}'] for i in range(n)], 1).astype(np.float32)
    means = np.stack([rows['x'], rows['y'], rows['z']], 1).astype(np.float32)
    return means, stack('scale_', 3), stack('rot_', 4), rows['opacity'].astype(np.float32)


def render_depth(gs, viewmat, K, w, h):
    import torch
    from gsplat import rasterization
    means, log_scales, quats, opacity_logit = gs
    depth, alpha, _ = rasterization(
        means=means, quats=torch.nn.functional.normalize(quats, dim=-1), scales=torch.exp(log_scales),
        opacities=torch.sigmoid(opacity_logit), colors=torch.zeros_like(means),
        viewmats=viewmat[None], Ks=K[None], width=w, height=h, render_mode='ED', packed=False)
    d = depth[0, ..., 0]
    return torch.where(alpha[0, ..., 0] > 0.5, d, torch.full_like(d, float('inf')))


def vote(means_np, gs_np, viewmats, Ks, label_maps, track_index, k, depth_tol):
    """Each splat votes in every frame where its center is the visible surface. Returns votes + views."""
    import torch
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    means = torch.from_numpy(means_np).to(dev)
    gs = tuple(torch.from_numpy(a).to(dev) for a in gs_np) if gs_np is not None else None
    n = len(means)
    seen = torch.zeros(n, dtype=torch.int32, device=dev)
    keys = []
    max_id = max(int(m.max()) for m in label_maps) if label_maps else 0
    lut = torch.full((max(max_id, max(track_index, default=0)) + 2,), -1, dtype=torch.int64, device=dev)
    for tid, idx in track_index.items():
        lut[tid] = idx
    ones = torch.ones(n, 1, device=dev)
    for f, labels_np in enumerate(label_maps):
        h, w = labels_np.shape
        V = torch.from_numpy(viewmats[f]).to(dev)
        K = torch.from_numpy(Ks[f]).to(dev)
        cam = torch.cat([means, ones], 1) @ V.T
        z = cam[:, 2]
        uv = cam[:, :3] @ K.T
        u = (uv[:, 0] / z.clamp(min=1e-6)).long()
        v = (uv[:, 1] / z.clamp(min=1e-6)).long()
        ok = (z > 1e-3) & (u >= 0) & (u < w) & (v >= 0) & (v < h)
        depth = render_depth(gs, V, K, w, h)
        idx = torch.nonzero(ok, as_tuple=True)[0]
        surf = depth[v[idx], u[idx]]
        visible = idx[torch.isfinite(surf) & ((z[idx] - surf).abs() <= depth_tol * surf + 1e-3)]
        seen[visible] += 1
        lab = torch.from_numpy(labels_np.astype(np.int64)).to(dev)
        tid = lab[v[visible], u[visible]]
        hit = tid >= 0
        obj = lut[tid[hit]]
        keep = obj >= 0
        keys.append(visible[hit][keep] * k + obj[keep])
    keys = torch.cat(keys) if keys else torch.zeros(0, dtype=torch.int64, device=dev)
    uniq, counts = torch.unique(keys, return_counts=True)
    return uniq.cpu().numpy(), counts.cpu().numpy(), seen.cpu().numpy()


def assign(uniq, counts, seen, n, k, min_views, min_fraction):
    splat = uniq // k
    obj = uniq % k
    order = np.lexsort((-counts, splat))  # per splat, highest vote first
    splat, obj, counts = splat[order], obj[order], counts[order]
    first = np.ones(len(splat), bool)
    first[1:] = splat[1:] != splat[:-1]
    s, o, c = splat[first], obj[first], counts[first]
    good = (c >= min_views) & (c >= min_fraction * seen[s])
    labels = np.full(n, -1, np.int32)
    labels[s[good]] = o[good]
    return labels


def robust_box(points):
    lo, hi = np.percentile(points, [2, 98], axis=0)
    return lo, hi


def box_overlap(a, b):
    """(IoU, intersection / smaller box) of two axis-aligned boxes."""
    lo, hi = np.maximum(a[0], b[0]), np.minimum(a[1], b[1])
    inter = np.prod(np.clip(hi - lo, 0, None))
    vol = lambda box: np.prod(np.clip(box[1] - box[0], 0, None))
    return inter / max(vol(a) + vol(b) - inter, 1e-12), inter / max(min(vol(a), vol(b)), 1e-12)


# ---------------------------------------------------------------- preview

def contact_sheet(path, objects, frames, label_maps, track_lists, cell=256, columns=5):
    from PIL import Image, ImageDraw
    if not objects:
        return
    rows = (len(objects) + columns - 1) // columns
    sheet = Image.new('RGB', (columns * cell, rows * (cell + 22)), (24, 24, 24))
    draw = ImageDraw.Draw(sheet)
    for n, ob in enumerate(objects):
        f = ob['best_frame_index']
        mask = np.isin(label_maps[f], track_lists[ob['id']])
        img = frames[f].copy()
        img[~mask] = (img[~mask] * 0.35).astype(np.uint8)
        ys, xs = np.nonzero(mask)
        if len(xs):
            pad = 20
            x0, x1 = max(xs.min() - pad, 0), min(xs.max() + pad, img.shape[1])
            y0, y1 = max(ys.min() - pad, 0), min(ys.max() + pad, img.shape[0])
            img = img[y0:y1, x0:x1]
        tile = Image.fromarray(img)
        tile.thumbnail((cell, cell))
        x, y = (n % columns) * cell, (n // columns) * (cell + 22)
        sheet.paste(tile, (x + (cell - tile.width) // 2, y + (cell - tile.height) // 2))
        draw.text((x + 6, y + cell + 4), f"{ob['id']:02d} {ob['label']} ({ob['score']:.2f})", fill=(230, 230, 230))
    sheet.save(path, quality=88)


# ---------------------------------------------------------------- main

def main():
    p = argparse.ArgumentParser()
    p.add_argument('--work', default='/content/work')
    p.add_argument('--scene', default='/content/scene/scene.ply')
    p.add_argument('--out', default='/content/objects')
    p.add_argument('--prompts', default=DEFAULT_PROMPTS)
    p.add_argument('--min-score', type=float, default=0.5)
    p.add_argument('--min-frames', type=int, default=3)
    p.add_argument('--min-views', type=int, default=3)
    p.add_argument('--min-fraction', type=float, default=0.5)
    p.add_argument('--min-splats', type=int, default=100)
    p.add_argument('--depth-tol', type=float, default=0.05)
    p.add_argument('--max-area-frac', type=float, default=0.35, help='drop tracks whose mask ever covers more of the frame than this')
    p.add_argument('--contain', type=float, default=0.7, help='merge groups when box overlap / smaller box exceeds this')
    p.add_argument('--no-cache', action='store_true')
    p.add_argument('--window', type=int, default=40)
    p.add_argument('--overlap', type=int, default=8)
    p.add_argument('--precision', default='float16', choices=['float16', 'float32'])
    args = p.parse_args()

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    for old in out.glob('*'):
        old.unlink()
    training = Path(json.loads((Path(args.work) / 'camera.json').read_text())['training'])
    prompts = [t.strip() for t in args.prompts.split(',') if t.strip()]
    frames, viewmats, Ks, _, _ = load_dataset(training)
    log(f'Segmenting {len(frames)} frames with {len(prompts)} prompts')

    t0 = time.time()
    cache = Path(args.work) / 'sam3_cache.pkl'
    key = {'prompts': prompts, 'window': args.window, 'overlap': args.overlap, 'n': len(frames)}
    loaded = None
    if cache.exists() and not args.no_cache:
        saved = pickle.loads(cache.read_bytes())
        if saved['key'] == key:
            loaded = saved['data']
            log('Reusing cached SAM 3 results (use --no-cache to redo)')
    if loaded is None:
        try:
            loaded = run_sam3(frames, prompts, args.precision, args.window, args.overlap)
        except FloatingPointError:
            log('float16 gave invalid scores; rerunning SAM 3 in float32')
            loaded = run_sam3(frames, prompts, 'float32', args.window, args.overlap)
        cache.write_bytes(pickle.dumps({'key': key, 'data': loaded}))
    label_maps, tracks, pair_iou = loaded
    log(f'SAM 3 found {len(tracks)} tracks in {(time.time() - t0) / 60:.1f} min')

    # Keep confident, persistent tracks; merge tracks that are the same object in 2D.
    frame_px = label_maps[0].size
    too_big = {t for t, info in tracks.items() if info.get('max_area', info['best_area']) > args.max_area_frac * frame_px}
    keep = {t for t, info in tracks.items() if info['score'] >= args.min_score and info['frames'] >= args.min_frames
            and t not in too_big}
    log(f'Dropped {len(too_big)} tracks covering >{args.max_area_frac:.0%} of the frame')
    uf = UnionFind()
    for t in keep:
        uf.find(t)
    for (a, b), iou in pair_iou.items():
        if a in keep and b in keep:
            uf.union(a, b)
    groups: dict[int, list[int]] = {}
    for t in keep:
        groups.setdefault(uf.find(t), []).append(t)
    group_list = list(groups.values())
    track_index = {t: g for g, members in enumerate(group_list) for t in members}

    rows = read_ply(Path(args.scene))
    gs = splat_params(rows)
    t1 = time.time()
    k = max(len(group_list), 1)
    uniq, counts, seen = vote(gs[0], gs, viewmats, Ks, label_maps, track_index, k, args.depth_tol)
    labels = assign(uniq, counts, seen, len(rows), k, args.min_views, args.min_fraction)
    log(f'Voting done in {(time.time() - t1):.0f} s')

    # Merge same-prompt groups that occupy the same 3D box (track lost and re-found).
    means = gs[0]
    boxes = {g: robust_box(means[labels == g]) for g in range(len(group_list)) if (labels == g).sum() >= args.min_splats}
    uf3 = UnionFind()
    names = {g: {tracks[t]['prompt'] for t in group_list[g]} for g in boxes}
    gids = sorted(boxes)
    for i, a in enumerate(gids):
        uf3.find(a)
        for b in gids[i + 1:]:
            iou3, inside = box_overlap(boxes[a], boxes[b])
            if names[a] & names[b] and (iou3 > 0.5 or inside > args.contain):
                uf3.union(a, b)
    final = sorted({uf3.find(g) for g in gids})
    remap = np.full(len(group_list) + 1, -1, np.int32)
    for new, root in enumerate(final):
        for g in gids:
            if uf3.find(g) == root:
                remap[g] = new
    labels = np.where(labels >= 0, remap[labels], -1).astype(np.int32)

    objects, track_lists = [], {}
    for new, root in enumerate(final):
        members = [t for g in gids if uf3.find(g) == root for t in group_list[g]]
        pts = means[labels == new]
        lo, hi = robust_box(pts)
        best = max(members, key=lambda t: tracks[t]['best_area'])
        prompt_scores = {}
        for t in members:
            prompt_scores[tracks[t]['prompt']] = max(prompt_scores.get(tracks[t]['prompt'], 0), tracks[t]['score'])
        label = max(prompt_scores, key=prompt_scores.get)
        slug = re.sub(r'[^a-z0-9]+', '_', label.lower()).strip('_')
        ply_name = f'object_{new:02d}_{slug}.ply'
        write_ply(out / ply_name, rows[labels == new])
        track_lists[new] = members
        objects.append({
            'id': new, 'label': label, 'prompts': sorted(prompt_scores), 'score': round(max(prompt_scores.values()), 3),
            'splats': int(len(pts)), 'center': [round(float(x), 4) for x in pts.mean(0)],
            'box_min': [round(float(x), 4) for x in lo], 'box_max': [round(float(x), 4) for x in hi],
            'frames_seen': int(sum(tracks[t]['frames'] for t in members)),
            'best_frame_index': int(tracks[best]['best_frame']), 'ply': ply_name, 'sam3_track_ids': members,
        })

    np.save(out / 'labels.npy', labels)
    write_ply(out / 'scene_without_objects.ply', rows[labels < 0])
    (out / 'objects.json').write_text(json.dumps({
        'scene': str(args.scene), 'units': 'scene units (camera path ~3 units; not meters)',
        'prompts': prompts, 'settings': {k: v for k, v in vars(args).items() if k not in ('prompts',)},
        'objects': objects}, indent=1))
    contact_sheet(out / 'preview.jpg', objects, frames, label_maps, track_lists)
    labeled = int((labels >= 0).sum())
    log(f'DONE {len(objects)} objects, {labeled:,} of {len(rows):,} splats labeled '
        f'({(time.time() - t0) / 60:.1f} min total)')


if __name__ == '__main__':
    main()

In [ ]:
#@title 4 · Add your room video
#@markdown Paste a Google Drive **link** (also works for files shared with you) or a path like `MyDrive/room.mov`.<br>
#@markdown Leave blank to pick a file from your computer instead.
VIDEO = "" #@param {type:"string"}
import os, re, glob, shutil
os.makedirs('/content/input', exist_ok=True)
for old in glob.glob('/content/input/*'):
    os.remove(old)
v = VIDEO.strip()
m = re.search(r'/d/([-\w]{20,})|[?&]id=([-\w]{20,})', v) or re.fullmatch(r'()([-\w]{25,})', v)
try:
    if m:
        from google.colab import auth
        auth.authenticate_user()
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        drive_api = build('drive', 'v3', cache_discovery=False)
        fid = m.group(1) or m.group(2)
        meta = drive_api.files().get(fileId=fid, fields='name,size', supportsAllDrives=True).execute()
        dest = os.path.join('/content/input', meta['name'])
        with open(dest, 'wb') as f:
            dl = MediaIoBaseDownload(f, drive_api.files().get_media(fileId=fid, supportsAllDrives=True), chunksize=64 * 2**20)
            finished = False
            while not finished:
                _, finished = dl.next_chunk()
    elif v:
        from google.colab import drive
        drive.mount('/content/drive')
        rel = v.removeprefix('/content/drive/').lstrip('/')
        src = os.path.join('/content/drive', rel if rel.startswith(('MyDrive', 'Shareddrives')) else 'MyDrive/' + rel)
        shutil.copy(src, '/content/input/')
    else:
        from google.colab import files
        for name, data in files.upload().items():
            open(os.path.join('/content/input', name), 'wb').write(data)
except Exception as e:
    print(f'download error: {e!r}')
inputs = sorted(glob.glob('/content/input/*'))
size = sum(os.path.getsize(p) for p in inputs) / 2**20
print(f'✅ STEP 4 DONE: {len(inputs)} file(s), {size:.0f} MB: {[os.path.basename(p) for p in inputs]}' if inputs
      else '❌ STEP 4 FAILED: no file added')

In [ ]:
#@title 5 · Align the views and train the 3D scene (~20–30 min)
STEPS = 7000 #@param {type:"integer"}
import subprocess, time, os, glob, torch
major, minor = torch.cuda.get_device_capability(0)
env = {**os.environ, 'TORCH_CUDA_ARCH_LIST': f'{major}.{minor}', 'PYTHONUNBUFFERED': '1'}
open('/content/progress.log', 'w').close()
inputs = sorted(glob.glob('/content/input/*'))
with open('/content/run.log', 'w') as full_log:
    proc = subprocess.Popen(['python', '/content/colab_room.py', *inputs, '--repo', '/content/projection',
                             '--out', '/content/scene', '--steps', str(STEPS)],
                            stdout=full_log, stderr=subprocess.STDOUT, env=env)
    shown = 0
    while True:
        done = proc.poll() is not None
        lines = open('/content/progress.log').read().splitlines()
        for line in lines[shown:]:
            print(line, flush=True)
        shown = len(lines)
        if done:
            break
        time.sleep(10)
if proc.returncode == 0 and os.path.isfile('/content/scene/scene.ply'):
    mb = os.path.getsize('/content/scene/scene.ply') / 2**20
    print(f'✅ STEP 5 DONE: scene.ply {mb:.0f} MB')
else:
    tail = open('/content/run.log').read().splitlines()
    print('\n'.join(l for l in tail if not l.startswith(('I20', 'W20')))[-3000:])
    print(f'❌ STEP 5 FAILED (exit {proc.returncode}). Full log: /content/run.log')

In [ ]:
#@title 6 · Save to / load from Google Drive
#@markdown **save** after step 5 so a Colab reset doesn't cost you alignment + training.<br>
#@markdown **load** in a fresh session (after steps 1–3) to skip steps 4–5.
ACTION = "save" #@param ["save", "load"]
RUN_NAME = "room1" #@param {type:"string"}
import os, subprocess
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
folder = f'/content/drive/MyDrive/room_window/{RUN_NAME}'
archive = f'{folder}/run.tar'
paths = ['content/work/camera.json', 'content/work/dataset/undistorted', 'content/scene', 'content/objects']
if ACTION == 'save':
    os.makedirs(folder, exist_ok=True)
    present = [p for p in paths if os.path.exists('/' + p)]
    r = subprocess.run(['tar', '-cf', archive, '-C', '/', *present], capture_output=True, text=True)
    ok = r.returncode == 0
    msg = f'saved {", ".join(os.path.basename(p) for p in present)} → {archive} ({os.path.getsize(archive) / 2**20:.0f} MB)' if ok else r.stderr[-500:]
else:
    r = subprocess.run(['tar', '-xf', archive, '-C', '/'], capture_output=True, text=True)
    ok = r.returncode == 0 and os.path.isfile('/content/scene/scene.ply')
    msg = f'loaded {archive}' if ok else (r.stderr[-500:] or 'scene.ply missing in archive')
print(f'✅ STEP 6 DONE: {msg}' if ok else f'❌ STEP 6 FAILED: {msg}')

In [ ]:
#@title 7 · Find interactable objects with SAM 3 (~5–15 min)
#@markdown Needs a Colab secret named `HF_TOKEN` (key icon, left sidebar) from a Hugging Face account with SAM 3 access.<br>
#@markdown Edit the list to change what counts as interactable (comma-separated).
PROMPTS = "mug, cup, bottle, bowl, plate, book, laptop, keyboard, phone, remote control, lamp, pillow, blanket, chair, stool, backpack, bag, box, basket, plant, vase, picture frame, clock, speaker, shoe, toy" #@param {type:"string"}
MIN_SCORE = 0.5 #@param {type:"number"}
import os, subprocess, time
from google.colab import userdata
try:
    token = userdata.get('HF_TOKEN')
except Exception as e:
    token = None
    print(f'Could not read HF_TOKEN secret: {e!r}')
if token:
    env = {**os.environ, 'HF_TOKEN': token, 'PYTHONUNBUFFERED': '1'}
    open('/content/progress.log', 'w').close()
    with open('/content/segment.log', 'w') as full_log:
        proc = subprocess.Popen(['python', '/content/segment_objects.py', '--work', '/content/work',
                                 '--scene', '/content/scene/scene.ply', '--out', '/content/objects',
                                 '--prompts', PROMPTS, '--min-score', str(MIN_SCORE)],
                                stdout=full_log, stderr=subprocess.STDOUT, env=env)
        shown = 0
        while True:
            done = proc.poll() is not None
            lines = open('/content/progress.log').read().splitlines()
            for line in lines[shown:]:
                print(line, flush=True)
            shown = len(lines)
            if done:
                break
            time.sleep(10)
    if proc.returncode == 0 and os.path.isfile('/content/objects/objects.json'):
        import json
        from IPython.display import Image, display
        objs = json.load(open('/content/objects/objects.json'))['objects']
        for o in objs:
            print(f"  {o['id']:02d} {o['label']:<16} score {o['score']:.2f}  {o['splats']:>7,} splats  seen in {o['frames_seen']} frames")
        if os.path.isfile('/content/objects/preview.jpg'):
            display(Image('/content/objects/preview.jpg'))
        print(f'✅ STEP 7 DONE: {len(objs)} objects in /content/objects (rerun step 6 "save" to keep them)')
    else:
        tail = open('/content/segment.log').read()
        print(tail[-3000:])
        hint = ' (403/gated: accept SAM 3 terms on huggingface.co/facebook/sam3 with the same account as the token)' if ('403' in tail or 'gated' in tail.lower()) else ''
        print(f'❌ STEP 7 FAILED (exit {proc.returncode}){hint}. Full log: /content/segment.log')
else:
    print('❌ STEP 7 FAILED: add a Colab secret named HF_TOKEN and allow this notebook to access it')

In [ ]:
#@title 8 · Open the walkthrough viewer
import sys, threading, socket, logging
sys.path.insert(0, '/content/projection')
from pipeline import walkthrough as wt
scene_id = open('/content/scene/scene_id.txt').read().strip()

def port_open(port):
    with socket.socket() as s:
        return s.connect_ex(('127.0.0.1', port)) == 0

if not port_open(8181):
    logging.getLogger('werkzeug').setLevel(logging.ERROR)
    app = wt.create_app(7000)
    app.config['TRUSTED_HOSTS'] = None  # Colab's proxy uses its own host name
    threading.Thread(target=lambda: app.run(host='127.0.0.1', port=8181, use_reloader=False), daemon=True).start()
    import time; time.sleep(2)

from google.colab.output import eval_js
base = eval_js('google.colab.kernel.proxyPort(8181)')
url = f"{base.rstrip('/')}/room/{scene_id}"
from IPython.display import HTML, display
display(HTML(f'<a href="{url}" target="_blank" style="font-size:18px">▶ Open walkthrough</a>'))
print('Controls: 2 = Fly mode, WASD = move, drag/G = look, Q/E = up/down, R = reset')
print(f'✅ STEP 8 DONE: {url}' if port_open(8181) else '❌ STEP 8 FAILED: viewer server did not start')

In [ ]:
#@title 9 · (Optional) Download the scene + objects
import shutil
from google.colab import files
import os
os.makedirs('/content/export', exist_ok=True)
for d in ('scene', 'objects'):
    if os.path.isdir(f'/content/{d}'):
        shutil.copytree(f'/content/{d}', f'/content/export/{d}', dirs_exist_ok=True)
shutil.make_archive('/content/room_scene', 'zip', '/content/export')
files.download('/content/room_scene.zip')
print('✅ STEP 9 DONE: room_scene.zip (scene/ + objects/). You can also drag any .ply into https://superspl.at/editor')